# Vocabulary expansion on SmolLM2-135M → Greek (Colab driver)

Runs the whole study on a **free Colab T4**: data preparation → tokenizer extension + QA →
throughput calibration → baseline / Exp1 / Exp2 with **matched tokens and steps** → evaluation
→ results table and plots → combined report and export. Artifacts are cached on Google Drive, or on
the VM disk if Drive is off or its mount fails (§1). Every run resumes from its last checkpoint after a
disconnect, as long as the artifacts survive.

The committed results in `results/t4_run/` came from these steps on the VM disk (the Drive mount timed
out), with the ablation command that is now the default §7 cell. `results/t4_run/README.md` lists where
that run differed from these cells.

| stage | time on the committed T4 run |
|---|---|
| data prep (stream, clean, dedup, split) | ~4 min, once |
| continued-BPE tokenizer (60 MB) + QA | ~2 min, once |
| calibration (20 steps) | ~3 min, once |
| baseline eval (BPB, cloze, generation) | ~5.5 min |
| Exp1 / Exp2 (training ≤ 30 min + dev evals, checkpoints, final eval) | ~39 / ~41 min (Exp2 also evaluates its step-0 state) |
| whole §6 plan (baseline + Exp1 + Exp2) | ~86 min |
| §6b interpolation (optional) | ~2 min per run |
| §7 bytes-matched Exp2 (optional) | ~12 min |

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + persistent storage
Either clone the repository (set `REPO_URL`) or upload the repo folder to Drive as `MyDrive/vocabx/repo`.

- `USE_DRIVE = True` (default): artifacts (data, tokenizers, checkpoints, results) go to
  `MyDrive/vocabx/artifacts` and survive a runtime reset. If the mount fails, the cell stops.
- `USE_DRIVE = "auto"`: as above, but if the mount fails or times out, use the VM disk instead.
- `USE_DRIVE = False`: artifacts go to the VM disk at `/content/vocabx_store/artifacts` and are **lost
  when the runtime is recycled**, so run §8 (export) before you leave. A run on the VM disk does not
  resume runs saved on Drive.

The committed `results/t4_run` used the VM disk (its README lists how that run's setup differed).

In [ ]:
REPO_URL = "https://github.com/nevasini1/smollm2-greek-vocab-expansion.git"   # set to "" to use a copy uploaded to Drive instead
USE_DRIVE = True   # True: Drive (a failed mount stops here) | "auto": Drive, else the VM disk | False: VM disk (lost on recycle)

import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/vocabx"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        USE_DRIVE = True
    except Exception as e:   # e.g. "mount failed" when the permission window times out
        if USE_DRIVE != "auto":
            raise RuntimeError(f"Drive mount failed ({e!r}). Re-run this cell, or set USE_DRIVE = 'auto' or False "
                               "to use the VM disk (runs saved on Drive are then not resumed).") from e
        print(f"WARNING: Drive mount failed ({e!r}); using the VM disk. Runs saved on Drive will NOT be resumed.")
        USE_DRIVE = False
if not USE_DRIVE and not REPO_URL:
    raise ValueError("REPO_URL is empty and Drive is not mounted, so there is no copy of the repo to use.")
STORE = DRIVE_ROOT if USE_DRIVE else "/content/vocabx_store"
os.makedirs(STORE, exist_ok=True)
%cd /content
if REPO_URL:   # (re)clone; leaving the folder first keeps re-runs of this cell safe
    !rm -rf /content/vocabx && git clone -q {REPO_URL} /content/vocabx
else:
    !rm -rf /content/vocabx && cp -r {DRIVE_ROOT}/repo /content/vocabx
%cd /content/vocabx
!pip install -q -e .
if "/content/vocabx/src" not in sys.path:   # the running kernel never reads the editable install; §8 imports vocabx
    sys.path.insert(0, "/content/vocabx/src")
WORK = f"{STORE}/artifacts"
W = f"--set workdir={WORK}"
print("artifacts ->", WORK, "(Google Drive)" if USE_DRIVE else "(VM disk: run section 8 before the runtime is recycled)")

In [ ]:
# Optional: a Hugging Face token raises Hub rate limits (all default datasets are ungated).
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 2. Data: training pools + fixed, decontaminated eval sets
Greek: Wikipedia (el) + FineWeb-2 ell_Grek (75 MB each) for training; held-out Wikipedia docs, the
official FineWeb-2 test split and Belebele/FLORES passages for evaluation. English: FineWeb-Edu
(SmolLM2's own distribution; also the replay pool), BBC News Jan–Jun 2025 (after SmolLM2's release)
and the parallel Belebele passages. Held-out docs are drawn first; training docs that share a
paragraph or are MinHash near-duplicates of any held-out doc are dropped.

The command exits as soon as the artifact is written. (Before cd617f8, streaming threads could keep it
alive after `data artifact ready`; interrupting it at that point is safe, because `manifest.json` is
written last.)

In [ ]:
!python -m vocabx prepare --config configs/exp2_extend.yaml {W}

## 3. Extend the tokenizer (continued BPE, +8,000 Greek tokens) and run the QA harness
Checks: original ids/merges untouched · English and any non-Greek text tokenize identically ·
`decode(encode(x)) == x` · every new token reachable (self-tokenization test) · compression and
vocab-size curve from merge prefixes (1k, 2k, 4k, 8k; capped at the number of new tokens).

In [ ]:
!python -m vocabx tokenizer --config configs/exp2_extend.yaml {W}

## 4. Fix the matched step budget ONCE (calibrated on the slower arm, Exp2)
`calibrate` measures throughput on this GPU and writes `artifacts/budget.json` with
`max_steps = min(steps that fit 27 min, steps the Greek pool covers without repetition)`.
Every later run, in this and any future session, reads that file, so a reconnect can never
change the budget between arms. Delete `budget.json` (or pass `--force`) only to start over.

In [ ]:
!python -m vocabx calibrate --config configs/exp2_extend.yaml {W} --steps 20 --minutes 27
import json
budget = json.load(open(f"{WORK}/budget.json"))
cal = budget["from_calibration"]
print({k: cal[k] for k in ["tokens_per_second", "suggested_max_steps", "pool_cap_steps", "recommended_max_steps",
                           "peak_memory_gb", "greek_MB_available", "sdpa"]})
print("matched budget: max_steps =", budget["max_steps"], f"({budget['max_steps'] * cal['tokens_per_step'] / 1e6:.1f}M tokens per arm)")

## 5. (Optional) LR probe on dev data, both arms, 150 steps each (~1 h total)
Pre-registered rule: the largest LR whose English-dev loss is ≤ 1/3 of its Greek-dev gain, both
measured from the ORIGINAL model and averaged over arms. The choice is written to
`artifacts/lr_choice.json`, and all later trained runs pick it up automatically. Run it before §6.

In [ ]:
RUN_PROBE = False
if RUN_PROBE:
    # curves only runs if the probe plan succeeded, and refuses to decide on incomplete runs
    !python -m vocabx plan configs/plans/lr_probe.yaml {W} && python -m vocabx curves --workdir {WORK} --runs probe_exp1_lr1e-4 probe_exp2_lr1e-4 probe_exp1_lr3e-4 probe_exp2_lr3e-4 probe_exp1_lr1e-3 probe_exp2_lr1e-3

## 6. Main comparison: baseline, Exp1, Exp2 (matched tokens and steps)
Resumable: if Colab disconnects, re-run this cell; finished runs are skipped and the
interrupted one resumes from its last checkpoint
(on the VM disk this works only while the same runtime is alive).

In [ ]:
!python -m vocabx plan configs/plans/main.yaml {W}

In [ ]:
from IPython.display import Markdown, Image, display
R = f"{WORK}/results/main"
display(Markdown(open(f"{R}/results.md").read()))
for img in ["learning_curves.png", "tradeoff_trajectories.png", "english_mass_on_new_tokens.png"]:
    if os.path.exists(f"{R}/{img}"):
        display(Image(f"{R}/{img}"))

## 6b. (Optional) Weight-interpolation frontiers (eval only, ~2 min per run on dev sets)
θ(α) = (1−α)·θ_start + α·θ_final for each arm. θ_start is the expanded step-0 model for Exp2. Comparing frontiers shows English given up per unit of Greek gained, independent of where the 30-min run stopped.

In [ ]:
DEV = "wikipedia_el:dev fineweb2_el:dev fineweb_edu_en:dev"
!python -m vocabx interpolate --config configs/exp1_finetune.yaml {W} --sets {DEV}
!python -m vocabx interpolate --config configs/exp2_extend.yaml {W} --sets {DEV} --plot-runs exp1_finetune exp2_extend
display(Image(f"{WORK}/results/interpolation_frontiers.png"))

## 7. Ablations and seeds (optional)
The default cell runs what the writeup reports: Exp2 stopped once it has read as many Greek bytes as
Exp1 (159 steps on the T4, ~12 min), with BPB-only evaluation (no cloze, generation or step-0 eval;
dev eval every 100 steps). The commented line is next on the ranked list: the 20% English-replay pair
with full evaluation (two more ~40-min runs). Each ablation is one YAML in `configs/ablations/`.

In [ ]:
# As run for results/t4_run: the bytes-matched Exp2 only, BPB-only evaluation.
!python -m vocabx plan configs/plans/ablations.yaml {W} --only exp2_bytes_matched --set 'eval.downstream=[]' --set eval.generation=false --set eval.full_eval_at_step0=false --set train.eval_every=100
# Next on the ranked list (~1.5 h more): the 20% English-replay pair, full evaluation.
# !python -m vocabx plan configs/plans/ablations.yaml {W} --only exp1_replay20 exp2_replay20

In [ ]:
# Training-noise estimate (two more seeds per arm, common-random-number pairs):
# !python -m vocabx plan configs/plans/seeds.yaml {W}

In [ ]:
R = f"{WORK}/results/ablations"
if os.path.exists(f"{R}/results.md"):
    display(Markdown(open(f"{R}/results.md").read()))

## 8. Combined report and export
Builds the report the writeup uses (every model that was run plus the paired contrasts, written to
`results/combined/`), then packs everything worth keeping into one zip and downloads it:

- reports and plots
- budget and calibration
- data manifests
- tokenizers and their QA
- each run's config, summary, logs, dev curves and per-chunk eval records

Model weights (`runs/*/final/`), checkpoints, the training pools and the eval-set files are left out;
copy weights to Drive yourself if you need them. Unzip the download into `results/<name>/` in the repo,
as was done for `results/t4_run/`.

In [ ]:
import traceback
from IPython.display import Markdown, display
from vocabx.analysis.report import build_report

runs = [r for r in ["base", "exp1_finetune", "exp2_extend", "exp2_bytes_matched"]
        if os.path.exists(f"{WORK}/runs/{r}/summary.json")]
pairs = [p for p in [["exp2_extend", "exp1_finetune"], ["exp2_bytes_matched", "exp1_finetune"]]
         if all(r in runs for r in p)]
title = "SmolLM2-135M -> Greek on a Colab T4: baseline, Exp1, Exp2"
if "exp2_bytes_matched" in runs:
    title += ", and Exp2 matched on Greek bytes"
try:   # a report error must not stop the export below
    build_report(WORK, runs, "base", "exp1_finetune", out_dir=f"{WORK}/results/combined", pairs=pairs, title=title)
    display(Markdown(open(f"{WORK}/results/combined/results.md").read()))
except Exception:
    traceback.print_exc()

In [ ]:
import glob, zipfile

EXPORT = "/content/vocabx_results.zip"
patterns = ["budget.json", "lr_choice.json", "calibration/*", "data/*/manifest.json",
            "tokenizers/*/*.json", "results/**/*", "runs/**/*"]
files = sorted({p for pat in patterns for p in glob.glob(f"{WORK}/{pat}", recursive=True) if os.path.isfile(p)})
files = [p for p in files if "/final/" not in p and not os.path.basename(p).startswith("checkpoint.pt")]
with zipfile.ZipFile(EXPORT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, os.path.relpath(p, WORK))
print(f"{len(files)} files -> {EXPORT} ({os.path.getsize(EXPORT) / 1e6:.1f} MB)")

from google.colab import files as colab_files
colab_files.download(EXPORT)